# 09 — Pull PRIO-GRID

**Source:** PRIO-GRID v3  
**Provider:** Peace Research Institute Oslo (PRIO)  
**Access:** Direct download from grid.prio.org (registration required)  
**Coverage:** Global, 0.5° × 0.5° grid cells  
**Frequency:** Static (geographic invariants) + Annual time-varying  

## What this notebook pulls

PRIO-GRID is a unified spatial data structure used for **subnational conflict models**
(ViEWS, Muchlinski 2013, Fearon et al. 2021). It provides spatially-explicit predictors
at grid-cell level (0.5° ≈ 55 km at the equator).

| File | Contents |
|---|---|
| `priogrid_cellshp.zip` | Grid cell geometries + static geographic variables |
| `priogrid-yearly.csv` | Annual time-varying variables per cell |

### Key variables (from DatasetVariableSynthesis.md §1.2 PRIO-GRID)

**Static (geographic):**
- `xcoord`, `ycoord` — centroid coordinates
- `bdist1` — distance to nearest international border (km)
- `bdist2` — distance to nearest coast (km)
- `mountains_mean` — mean terrain ruggedness
- `forest_gc_per` — forest cover (%)
- `gwno` — Gleditsch-Ward country code

**Annual time-varying:**
- `pop_gpw_sum` — grid cell population (Gridded Population of the World)
- `nlights_calib_mean` — nighttime lights (DMSP-OLS; proxy for economic activity)
- `excluded_group` — excluded ethnic group present (from EPR)
- `petroleum_y` — petroleum extraction flag
- `drug_y` — drug crop cultivation flag
- `rainfall_sd` — rainfall variability (std dev from GPCP)
- `temp_sd` — temperature variability

## Access note
PRIO-GRID requires a free registration at https://grid.prio.org
After downloading, place the files as:
```
prio_files/priogrid_cell.csv     ← static variables
prio_files/priogrid-yearly.csv   ← annual time-varying variables
```
Alternatively, set `PRIO_STATIC_URL` and `PRIO_YEARLY_URL` if direct download links
are provided in your registration confirmation email.

## Required environment variables
```
ADLS_ACCOUNT_NAME  — Azure storage account name
ADLS_CONTAINER     — Container name (default: 'data')
PRIO_STATIC_URL    — (optional) direct download URL for static file
PRIO_YEARLY_URL    — (optional) direct download URL for yearly file
```

In [ ]:
import os
import io
import zipfile
import requests
import pandas as pd
from pathlib import Path
from datetime import datetime
from azure.identity import DefaultAzureCredential

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

# If PRIO-GRID sends you a direct download URL after registration, set these:
PRIO_STATIC_URL = os.getenv("PRIO_STATIC_URL", "")
PRIO_YEARLY_URL = os.getenv("PRIO_YEARLY_URL", "")

# Local fallback: files downloaded manually
PRIO_LOCAL_DIR   = Path("prio_files")
PRIO_STATIC_FILE = PRIO_LOCAL_DIR / "priogrid_cell.csv"
PRIO_YEARLY_FILE = PRIO_LOCAL_DIR / "priogrid-yearly.csv"

print(f"Run date: {RUN_DATE}")
print(f"Static URL set : {bool(PRIO_STATIC_URL)}")
print(f"Yearly URL set : {bool(PRIO_YEARLY_URL)}")
print(f"Local files    : static={PRIO_STATIC_FILE.exists()}, yearly={PRIO_YEARLY_FILE.exists()}")

## ADLS helper

In [ ]:
credential = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential": credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

## Load helper

Tries URL first, then local file.

In [ ]:
def load_prio_file(url: str, local_path: Path, label: str) -> pd.DataFrame | None:
    if url:
        print(f"Downloading PRIO-GRID {label} from URL...")
        resp = requests.get(url, timeout=300, stream=True)
        resp.raise_for_status()
        content = resp.content

        # Handle zip archives
        if resp.headers.get("content-type", "").startswith("application/zip") or url.endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(content)) as z:
                csv_name = next(
                    (n for n in z.namelist() if n.endswith(".csv")), None
                )
                if csv_name:
                    with z.open(csv_name) as f:
                        return pd.read_csv(f, low_memory=False)
                else:
                    print(f"  No CSV inside zip for {label}")
                    return None
        else:
            return pd.read_csv(io.BytesIO(content), low_memory=False)

    elif local_path.exists():
        print(f"Loading PRIO-GRID {label} from local file: {local_path}")
        return pd.read_csv(local_path, low_memory=False)

    else:
        print(
            f"\nWARNING: {label} not available.\n"
            f"  Register at https://grid.prio.org and either:\n"
            f"  • Set env var PRIO_{label.upper().replace('-','_')}_URL, or\n"
            f"  • Place downloaded CSV at {local_path}\n"
        )
        return None

## 1. Static grid variables

In [ ]:
STATIC_COLS = [
    "gid",             # PRIO-GRID cell ID (join key)
    "gwno",            # Gleditsch-Ward country code
    "xcoord",          # Longitude of centroid
    "ycoord",          # Latitude of centroid
    "bdist1",          # Distance to nearest international border (km)
    "bdist2",          # Distance to nearest coast (km)
    "mountains_mean",  # Terrain ruggedness (% mountainous)
    "forest_gc_per",   # Forest cover (%)
    "area",            # Cell area (km²)
]

df_static = load_prio_file(PRIO_STATIC_URL, PRIO_STATIC_FILE, "static")

if df_static is not None:
    df_static.columns = [c.lower().strip() for c in df_static.columns]
    available_static = [c for c in STATIC_COLS if c in df_static.columns]
    df_static = df_static[available_static].copy()
    print(f"Static grid: {len(df_static):,} cells | columns: {list(df_static.columns)}")
    df_static.head(3)

In [ ]:
if df_static is not None and not df_static.empty:
    write_parquet(df_static, f"raw/prio_grid/{RUN_DATE}/priogrid_static.parquet")

## 2. Annual time-varying variables

In [ ]:
YEARLY_COLS = [
    "gid",               # PRIO-GRID cell ID (join key)
    "year",
    "gwno",              # Country code (may change for cells near borders)
    "pop_gpw_sum",       # Grid-cell population (GPW)
    "nlights_calib_mean",# Mean nighttime light intensity
    "excluded_group",    # Excluded ethnic group present (from EPR)
    "petroleum_y",       # Petroleum extraction flag
    "drug_y",            # Drug crop cultivation flag
    "wdi_gdp",           # Grid-interpolated GDP
    "rainfall_sd",       # Rainfall variability (std dev)
    "temp_sd",           # Temperature variability (std dev)
    "ttime_mean",        # Mean travel time to nearest city (minutes)
]

df_yearly = load_prio_file(PRIO_YEARLY_URL, PRIO_YEARLY_FILE, "yearly")

if df_yearly is not None:
    df_yearly.columns = [c.lower().strip() for c in df_yearly.columns]
    available_yearly = [c for c in YEARLY_COLS if c in df_yearly.columns]
    df_yearly = df_yearly[available_yearly].copy()

    if "year" in df_yearly.columns:
        df_yearly["year"] = pd.to_numeric(df_yearly["year"], errors="coerce").astype("Int64")
    if "pop_gpw_sum" in df_yearly.columns:
        df_yearly["pop_gpw_sum"] = pd.to_numeric(df_yearly["pop_gpw_sum"], errors="coerce")
    if "nlights_calib_mean" in df_yearly.columns:
        df_yearly["nlights_calib_mean"] = pd.to_numeric(df_yearly["nlights_calib_mean"], errors="coerce")

    print(f"Yearly grid: {len(df_yearly):,} cell-years | {df_yearly['gid'].nunique()} cells")
    if "year" in df_yearly.columns:
        print(f"Years: {df_yearly['year'].min()}–{df_yearly['year'].max()}")
    df_yearly.head(3)

In [ ]:
if df_yearly is not None and not df_yearly.empty:
    write_parquet(df_yearly, f"raw/prio_grid/{RUN_DATE}/priogrid_yearly.parquet")

## 3. Country-level aggregates from PRIO-GRID

The panel model works at **country-year** level, not grid-cell level.
We aggregate the time-varying cell data to country-year means/sums
for the structural predictors that matter most at country scale.

In [ ]:
if df_yearly is not None and not df_yearly.empty and "gwno" in df_yearly.columns:
    agg_map = {}
    for col in ["pop_gpw_sum"]:
        if col in df_yearly.columns:
            agg_map[col] = "sum"
    for col in ["nlights_calib_mean", "wdi_gdp", "rainfall_sd", "temp_sd", "ttime_mean"]:
        if col in df_yearly.columns:
            agg_map[col] = "mean"
    for col in ["excluded_group", "petroleum_y", "drug_y"]:
        if col in df_yearly.columns:
            agg_map[col] = "max"  # 1 if any cell in country has the flag

    group_cols = [c for c in ["gwno", "year"] if c in df_yearly.columns]

    if agg_map and group_cols:
        df_prio_country = (
            df_yearly
            .groupby(group_cols, as_index=False)
            .agg(agg_map)
        )
        print(f"Country-year aggregates: {df_prio_country.shape}")
        write_parquet(df_prio_country, f"raw/prio_grid/{RUN_DATE}/priogrid_country_year_agg.parquet")
    else:
        print("Skipping country aggregation — missing required columns")
        df_prio_country = pd.DataFrame()
else:
    df_prio_country = pd.DataFrame()

## Summary

In [ ]:
print("=" * 55)
print("PRIO-GRID pull complete")
print("=" * 55)
if df_static is not None and not df_static.empty:
    print(f"  Static cells      : {len(df_static):,}")
if df_yearly is not None and not df_yearly.empty:
    print(f"  Yearly cell-years : {len(df_yearly):,}")
if not df_prio_country.empty:
    print(f"  Country-yr agg    : {len(df_prio_country):,}")
print()
print("ADLS paths written:")
print(f"  raw/prio_grid/{RUN_DATE}/priogrid_static.parquet")
print(f"  raw/prio_grid/{RUN_DATE}/priogrid_yearly.parquet")
print(f"  raw/prio_grid/{RUN_DATE}/priogrid_country_year_agg.parquet")